# Clustering: K-Means and Gaussian Mixture Models

This notebook walks through the from-scratch implementations in the `clustering` package:

1. Load four labeled synthetic datasets
2. Fit **K-Means**, **Soft K-Means**, and **GMM (EM)**
3. Compare against scikit-learn
4. Choose \(k\) with silhouette / BIC

See `docs/algorithms.md` for the math and design notes.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans as SkKMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.mixture import GaussianMixture as SkGaussianMixture

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from clustering import (
    GaussianMixtureModel,
    KMeans,
    SoftKMeans,
    clustering_report,
)
from clustering.model_selection import (
    elbow_curve,
    gmm_information_criteria,
    suggest_k,
)
from clustering.viz import plot_datasets, plot_kmeans_comparison, plot_soft_assignments

%matplotlib inline

## 1. Datasets

Each CSV has columns `x`, `y`, and ground-truth `cat` (3 classes, 1000 points each).

In [ ]:
names = ["mv", "unif", "mv2", "mv3"]
dataframes = [pd.read_csv(ROOT / "data" / f"{name}.csv", index_col=0) for name in names]
datasets = {
    name: (df[["x", "y"]].to_numpy(dtype=float), df["cat"].to_numpy())
    for name, df in zip(names, dataframes)
}

fig = plot_datasets(dataframes, titles=names)
plt.show()

## 2. Custom K-Means vs scikit-learn

Centers from the custom estimator (black ×) should land next to sklearn centers (red +).

In [ ]:
fig = plot_kmeans_comparison(dataframes, n_clusters=3, random_state=0)
plt.show()

rows = []
for name, (X, y) in datasets.items():
    custom = KMeans(n_clusters=3, random_state=0).fit(X)
    sk = SkKMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
    rows.append({
        "dataset": name,
        "custom_ari": adjusted_rand_score(y, custom.labels_),
        "sklearn_ari": adjusted_rand_score(y, sk.labels_),
        "custom_inertia": custom.inertia_,
        "sklearn_inertia": sk.inertia_,
    })
pd.DataFrame(rows)

,dataset,custom_ari,sklearn_ari,custom_inertia,sklearn_inertia
0,mv,0.618632,0.618632,2176.816993,2176.824130
1,unif,0.245617,0.245617,1767.383174,1767.384072
2,mv2,0.421936,0.462067,2615.840939,2556.855932
3,mv3,0.983101,0.983101,2539.650201,2539.650201


## 3. Soft K-Means and GMM

Soft methods shine when clusters overlap or stretch. On `mv2`, GMM typically beats hard K-Means by a wide ARI margin.

In [ ]:
rows = []
for name, (X, y) in datasets.items():
    soft = SoftKMeans(n_clusters=3, beta=0.3, random_state=0).fit(X)
    gmm = GaussianMixtureModel(n_components=3, random_state=0).fit(X)
    sk_gmm = SkGaussianMixture(n_components=3, random_state=0).fit(X)
    rows.append({
        "dataset": name,
        "soft_kmeans_ari": adjusted_rand_score(y, soft.labels_),
        "custom_gmm_ari": adjusted_rand_score(y, gmm.labels_),
        "sklearn_gmm_ari": adjusted_rand_score(y, sk_gmm.predict(X)),
        "custom_gmm_bic": gmm.bic(X),
    })
pd.DataFrame(rows)

,dataset,soft_kmeans_ari,custom_gmm_ari,sklearn_gmm_ari,custom_gmm_bic
0,mv,0.617154,0.842669,0.828550,15524.727215
1,unif,0.239329,0.572921,0.556856,11578.702483
2,mv2,0.423186,0.853584,0.851023,14209.182132
3,mv3,0.983101,0.992017,0.992017,18079.892620


In [ ]:
fig = plot_soft_assignments(dataframes, model="gmm", n_clusters=3, random_state=0)
plt.show()
fig = plot_soft_assignments(dataframes, model="kmeans", n_clusters=3, random_state=0)
plt.show()

## 4. Choosing the number of clusters

Use silhouette for K-Means and BIC for GMMs.

In [ ]:
X, y = datasets["mv3"]
elbow = pd.DataFrame(elbow_curve(X, range(1, 8), random_state=0))
ics = pd.DataFrame(gmm_information_criteria(X, range(1, 8), random_state=0))

print("Suggested k (silhouette):", suggest_k(elbow.to_dict("records"), "silhouette"))
print("Suggested k (BIC):", suggest_k(ics.to_dict("records"), "bic"))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(elbow["k"], elbow["inertia"], marker="o")
axes[0].set_title("K-Means elbow (inertia)")
axes[0].set_xlabel("k")
axes[1].plot(ics["k"], ics["bic"], marker="o", label="BIC")
axes[1].plot(ics["k"], ics["aic"], marker="s", label="AIC")
axes[1].set_title("GMM information criteria")
axes[1].set_xlabel("k")
axes[1].legend()
plt.tight_layout()
plt.show()

display(elbow)
display(ics)

Suggested k (silhouette): 1
Suggested k (BIC): 3


,k,inertia,n_iter,silhouette
0,1,18433.131953,2,NaN
1,2,10505.621713,4,0.476083
2,3,2539.650201,8,0.692719
3,4,2345.087065,10,0.537221
4,5,2191.876318,16,0.519837
5,6,1788.159243,15,0.451600
6,7,1469.962429,26,0.449732


,k,aic,bic,log_likelihood,n_iter
0,1,23771.436181,23801.468019,-11880.718091,3
1,2,19969.666856,20035.736899,-9973.833428,8
2,3,17977.784372,18079.892620,-8971.892186,8
3,4,17984.609965,18122.756419,-8969.304983,105
4,5,17984.042520,18158.227179,-8963.021260,300
5,6,17994.217100,18204.439965,-8962.108550,300
6,7,18000.460057,18246.721127,-8959.230028,300


## 5. Single-dataset report

Compact metrics for portfolio / debugging use.

In [ ]:
X, y = datasets["mv2"]
km = KMeans(n_clusters=3, random_state=0).fit(X)
gmm = GaussianMixtureModel(n_components=3, random_state=0).fit(X)

print("K-Means", clustering_report(X, y, km.labels_, km.cluster_centers_))
print("GMM", clustering_report(X, y, gmm.labels_, gmm.means_))

K-Means {'n_samples': 3000, 'n_clusters': 3, 'inertia': 2615.840938784385, 'silhouette': 0.4350834043012118, 'adjusted_rand_index': 0.4219360866333004, 'normalized_mutual_info': 0.5373396745702265}
GMM {'n_samples': 3000, 'n_clusters': 3, 'inertia': 2995.864487088909, 'silhouette': 0.36022670125179834, 'adjusted_rand_index': 0.8535835231851602, 'normalized_mutual_info': 0.8041405963072417}
